In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
from tqdm.notebook import tqdm  # Import tqdm specifically optimized for Jupyter

# 1. Load environment variables from your .env file
load_dotenv()
print("Connecting to PostgreSQL...")

# 2. Securely fetch the credentials
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

# 3. Construct the database URL and create the engine
db_url = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
engine = create_engine(db_url)

# 4. Ask PostgreSQL exactly how many rows exist so the progress bar knows what 100% is
count_query = "SELECT COUNT(*) FROM public.model_features;"
total_rows = pd.read_sql(count_query, engine).iloc[0, 0]

# 5. Fetch the data in chunks and update the progress bar
query = "SELECT * FROM public.model_features;"
chunksize = 100000 
chunks = [] # We will store our data chunks here temporarily

print(f"Fetching {total_rows:,} engineered features from local database...")

# Create the progress bar
with tqdm(total=total_rows, desc="Downloading Data") as pbar:
    # Read the data in chunks
    for chunk in pd.read_sql(query, engine, chunksize=chunksize):
        chunks.append(chunk)      # Save the chunk
        pbar.update(len(chunk))   # Move the progress bar forward

# 6. Stitch all the chunks together into one massive DataFrame
df = pd.concat(chunks, ignore_index=True)

print(f"Success! Loaded {len(df):,} rows into local memory.")
display(df.head())

Connecting to PostgreSQL...
Fetching 5,698,555 engineered features from local database...


Success! Loaded 5,698,555 rows into local memory.


,MONTH,DAY_OF_WEEK,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,DISTANCE,DEPARTURE_BLOCK,AIRLINE_HISTORIC_DELAY,ROUTE_HISTORIC_DELAY,IS_DELAYED
0,10,4,AA,10423,11057,1032,Afternoon,8.91,6.59,1
1,10,1,AA,10257,11057,646,Morning,8.91,1.56,0
2,10,2,AA,10257,11057,646,Afternoon,8.91,1.56,0
3,10,3,AA,10257,11057,646,Morning,8.91,1.56,0
4,10,4,AA,10693,12892,1797,Morning,8.91,5.39,0


In [2]:
from sklearn.model_selection import train_test_split

print("Starting preprocessing...")

# 1. Drop columns we don't need for the predictive model
# We drop ORIGIN_AIRPORT and DESTINATION_AIRPORT to keep the model fast, 
# relying on the ROUTE_HISTORIC_DELAY feature instead.
columns_to_drop = ['MONTH', 'DAY_OF_WEEK', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']
df_clean = df.drop(columns=columns_to_drop)

# 2. One-Hot Encode the categorical text columns (AIRLINE, DEPARTURE_BLOCK)
# This turns categories into binary numerical columns using pandas
df_encoded = pd.get_dummies(df_clean, columns=['AIRLINE', 'DEPARTURE_BLOCK'], drop_first=True)

# 3. Separate your Features (X) from your Target (y)
X = df_encoded.drop(columns=['IS_DELAYED'])
y = df_encoded['IS_DELAYED']

# 4. Split the data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Preprocessing complete!")
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

Starting preprocessing...
Preprocessing complete!
Training features shape: (4558844, 19)
Testing features shape: (1139711, 19)


In [3]:
from xgboost import XGBClassifier
import time

print("Initializing XGBoost Classifier...")
# We use standard default parameters to establish a strong baseline
model = XGBClassifier(
    random_state=42, 
    eval_metric='logloss' # Tells the model how to measure its own errors during training
)

print("Training the model on 80% of the data. Please wait...")
start_time = time.time()

# This single line is where the actual math and "learning" happens
model.fit(X_train, y_train)

end_time = time.time()
print(f"Success! Training complete in {round(end_time - start_time, 2)} seconds.")

Initializing XGBoost Classifier...
Training the model on 80% of the data. Please wait...
Success! Training complete in 40.23 seconds.


In [4]:
from sklearn.metrics import classification_report, confusion_matrix
import joblib

print("Evaluating model on the hidden test data...")

# 1. Ask the model to make predictions
y_pred = model.predict(X_test)

# 2. Print out the KPI metrics
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

# 3. Export the trained model to your hard drive
model_filename = 'xgboost_delay_model.joblib'
joblib.dump(model, model_filename)

print(f"\nBoom! Model fully trained, evaluated, and saved locally as: {model_filename}")

Evaluating model on the hidden test data...

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.82      1.00      0.90    936861
           1       0.54      0.00      0.00    202850

    accuracy                           0.82   1139711
   macro avg       0.68      0.50      0.45   1139711
weighted avg       0.77      0.82      0.74   1139711

--- Confusion Matrix ---
[[936739    122]
 [202707    143]]

Boom! Model fully trained, evaluated, and saved locally as: xgboost_delay_model.joblib


In [5]:
from xgboost import XGBClassifier
import time

print("Calculating class imbalance...")
# Count the 0s (On Time) and divide by the 1s (Delayed)
# Based on your confusion matrix, this will be roughly 4.6
imbalance_ratio = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
print(f"Imbalance Ratio: {round(imbalance_ratio, 2)} to 1")

print("\nInitializing Tuned XGBoost Classifier...")
model = XGBClassifier(
    random_state=42, 
    eval_metric='logloss',
    scale_pos_weight=imbalance_ratio  # <-- The magic parameter
)

print("Training the model. Please wait...")
start_time = time.time()
model.fit(X_train, y_train)
end_time = time.time()

print(f"Success! Training complete in {round(end_time - start_time, 2)} seconds.")

Calculating class imbalance...
Imbalance Ratio: 4.61 to 1

Initializing Tuned XGBoost Classifier...
Training the model. Please wait...
Success! Training complete in 43.08 seconds.


In [6]:
from sklearn.metrics import classification_report, confusion_matrix
import joblib

print("Evaluating model on the hidden test data...")

# 1. Ask the model to make predictions
y_pred = model.predict(X_test)

# 2. Print out the KPI metrics
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

# 3. Export the trained model to your hard drive
model_filename = 'xgboost_delay_model.joblib'
joblib.dump(model, model_filename)

print(f"\nBoom! Model fully trained, evaluated, and saved locally as: {model_filename}")

Evaluating model on the hidden test data...

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.89      0.60      0.71    936861
           1       0.26      0.66      0.37    202850

    accuracy                           0.61   1139711
   macro avg       0.57      0.63      0.54   1139711
weighted avg       0.78      0.61      0.65   1139711

--- Confusion Matrix ---
[[557687 379174]
 [ 69901 132949]]

Boom! Model fully trained, evaluated, and saved locally as: xgboost_delay_model.joblib
